# Dioptra-DINO: Foundation-Assisted Metric Depth on Edge Devices

This notebook trains **Dioptra-DINO** (~25.4M parameters) on dual NVIDIA T4 GPUs on Kaggle.

### Architecture Highlights:
- **Pre-trained Visual Backbone**: DINOv2-Small (`vits14`, 21.6M parameters pre-trained on 142M images).
- **Trivision Ray Positional Encoding**: Continuous optical ray unprojection from camera matrix $\mathbf{K}$.
- **Angular Residual Attention (ARA)**: Intrinsic geometric attention bias $\sin^2(\theta_{q, k})$ enforcing surface planarity.
- **Decoupled Scale Supervision**: Log-median metric scale loss $\mathcal{L}_{\text{scale}}$ ensuring accurate physical ranging in metres.

### Kaggle Setup Instructions:
1. **Settings (Right Sidebar)**:
   - **Accelerator**: GPU T4 x2 (or P100)
   - **Internet**: **ON** (allows automatic download of DINOv2 weights on first cell)
2. **+ Add Input**:
   - Attach dataset: **`pandrii000/dasvo-tartanair-rgb-d-validation-split`**
   - Attach code repository (contains `dioptra_dino.py`)

In [ ]:
# [1] Verify NVIDIA GPU Allocations
!nvidia-smi
import torch
print("CUDA available :", torch.cuda.is_available())
print("Device count   :", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}       : {torch.cuda.get_device_name(i)}")

In [ ]:
# [2] Locate dioptra_dino.py (supports Kaggle input upload OR auto-cloning from GitHub)
import glob, os, shutil

candidates = sorted(glob.glob("/kaggle/input/**/dioptra_dino.py", recursive=True))
if not candidates:
    if os.path.exists("dioptra_dino.py"):
        candidates = ["dioptra_dino.py"]
    elif os.path.exists("../dioptra_dino.py"):
        candidates = ["../dioptra_dino.py"]
    else:
        print("dioptra_dino.py not mounted in /kaggle/input. Auto-cloning from GitHub...")
        os.system("git clone https://github.com/SeranomTheGreat/dioptra.git /kaggle/working/dioptra_repo")
        git_src = "/kaggle/working/dioptra_repo/dioptra_dino.py"
        if os.path.exists(git_src):
            candidates = [git_src]

assert candidates, "Could not locate dioptra_dino.py! Please ensure Internet is ON or upload dioptra_dino.py."
script_src = candidates[-1]
script_dst = "/kaggle/working/dioptra_dino.py"
if os.path.abspath(script_src) != os.path.abspath(script_dst):
    shutil.copyfile(script_src, script_dst)
print(f"Ready! Using script at: {script_dst}")


In [ ]:
# [3] Run architectural sanity checks: Parameter audit, smoke test, and geometric unit tests
!python /kaggle/working/dioptra_dino.py --count
!python /kaggle/working/dioptra_dino.py --smoke
!python /kaggle/working/dioptra_dino.py --test

In [ ]:
# [4] Discover TartanAir dataset path
dataset_roots = []
for root in ['/kaggle/input/dasvo-tartanair-rgb-d-validation-split', '/kaggle/input']:
    if os.path.isdir(root):
        pngs = glob.glob(os.path.join(root, '**', '*.png'), recursive=True)
        if len(pngs) > 100:
            dataset_roots.append((root, len(pngs)))

assert dataset_roots, 'TartanAir dataset not detected! Please add "dasvo-tartanair-rgb-d-validation-split" in the right sidebar.'
data_path = sorted(dataset_roots, key=lambda x: x[1], reverse=True)[0][0]
print(f"Detected TartanAir dataset at: {data_path}")

In [ ]:
# [5] Launch Dioptra-DINO Training (15 Epochs on Dual T4)
# Backbone LR = 2e-5 (fine-tuning DINOv2 visual features)
# Geometric Head LR = 2e-4 (learning scale & ARA attention bias)
# Batch size = 8 per GPU, 4-step gradient accumulation (effective batch size = 32)
!python /kaggle/working/dioptra_dino.py \
    --train "$data_path" \
    --epochs 15 \
    --batch-size 8 \
    --lr-backbone 2e-5 \
    --lr-head 2e-4

In [ ]:
# [6] Verify output checkpoints
import os
print("Checkpoints in /kaggle/working/outputs_dino:")
for f in sorted(os.listdir('/kaggle/working/outputs_dino')):
    p = os.path.join('/kaggle/working/outputs_dino', f)
    size_mb = os.path.getsize(p) / (1024 * 1024)
    print(f"  {f} ({size_mb:.2f} MB)")